# 03 Network

The main spec uses `MIN_TX = 2`; `MIN_TX = 1` gives the sensitivity tables. Figures use the main spec.

Build the firm x product bipartite network from PQR and project it onto products.

Proximity between products i and j follows Hidalgo et al. (2007):
    phi_ij = C_ij / max(k_i, k_j)
where C_ij = number of firms that supplied both products and k_i = number of firms that
supplied product i. This equals min(P(i|j), P(j|i)) and keeps large products from
connecting to everything.

A firm "supplies" a product if it has at least MIN_TX PQR transactions for it, to drop
one-off purchases. Outputs go to data/processed/network/.


Community detection, null-model test, and centrality on the product network.

Steps
1. Louvain communities on the phi-weighted product network (best of 50 seeds; stability = mean
   ARI between seed runs).
2. Null model: rewire the firm x product bipartite network keeping every firm's and product's
   degree fixed (random edge swaps), re-project, re-run Louvain. Compare observed modularity
   and observed alignment with disease areas against 200 null draws.
3. Alignment of communities with official disease areas (NMI, ARI, cross-tab).
4. Centrality: strength (sum of phi), betweenness (distance = 1 - phi), eigenvector.


Figures for the product network (main spec MIN_TX=2).
f5: product space backbone (maximum spanning tree + strong ties), node color = disease area,
    shaded regions = Louvain communities, node size = spend.
f6: null-model test (observed vs 200 degree-preserving rewirings).
f7: disease-area composition of each community.

In [ ]:
%run 00_utils.ipynb

import numpy as np, pandas as pd, networkx as nx, matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

## Build

In [ ]:
def build_network(MIN_TX):

    pqr = load_clean("pqr_medicines")

    # keep firm-product pairs with at least min_tx transactions
    def bipartite_edges(transactions, min_tx=MIN_TX):
        links = transactions.groupby(["firm", "product"]).agg(n_tx=("pk", "size"), spend_usd=("cost_usd", "sum"),
                                                first_year=("year", "min"), last_year=("year", "max")).reset_index()
        return links[links.n_tx >= min_tx].reset_index(drop=True)

    links = bipartite_edges(pqr)
    links.to_csv(NETWORK / f"bipartite_edges_min{MIN_TX}.csv", index=False)

    product_nodes = pqr.groupby("product").agg(area=("area", "first"), spend_usd=("cost_usd", "sum"),
                                      n_tx=("pk", "size"), n_countries=("country", "nunique"),
                                      first_year=("year", "min")).reset_index()
    product_nodes["n_firms"] = product_nodes["product"].map(links.groupby("product")["firm"].nunique()).fillna(0).astype(int)
    firm_nodes = pqr.groupby("firm").agg(group=("group", "first"), hq_country=("hq_country", "first"),
                                   spend_usd=("cost_usd", "sum")).reset_index()
    firm_nodes["n_products"] = firm_nodes["firm"].map(links.groupby("firm")["product"].nunique()).fillna(0).astype(int)
    firm_nodes["n_areas"] = firm_nodes["firm"].map(links.merge(product_nodes[["product", "area"]]).groupby("firm")["area"].nunique()).fillna(0).astype(int)

    # project the bipartite network onto products and save its tables
    proximity_edges = project_products(links)
    proximity_edges.to_csv(NETWORK / f"product_edges_min{MIN_TX}.csv", index=False)
    product_nodes.to_csv(NETWORK / f"product_nodes_min{MIN_TX}.csv", index=False)
    firm_nodes.to_csv(NETWORK / f"firm_nodes_min{MIN_TX}.csv", index=False)

    network = nx.Graph()
    network.add_nodes_from(product_nodes[product_nodes.n_firms > 0]["product"])
    network.add_weighted_edges_from(proximity_edges[["source", "target", "phi"]].itertuples(index=False))
    isolates = [node for node in network if network.degree(node) == 0]
    print(f"MIN_TX={MIN_TX}")
    print(f"bipartite: {links.firm.nunique()} firms, {links['product'].nunique()} products, {len(links)} edges "
          f"(dropped {pqr.groupby(['firm','product']).ngroups - len(links)} firm-product pairs with < {MIN_TX} tx)")
    print(f"firms supplying 1 product: {(firm_nodes.n_products == 1).sum()} | >=10 products: {(firm_nodes.n_products >= 10).sum()}")
    print(f"product network: {network.number_of_nodes()} nodes, {network.number_of_edges()} edges, "
          f"density {nx.density(network):.3f}, isolates {len(isolates)}, components {nx.number_connected_components(network)}")
    print("phi quantiles:", proximity_edges.phi.quantile([.25, .5, .75, .9]).round(2).to_dict())
    print("isolated products:", isolates[:15])

## Analysis

In [ ]:
def analyze_network(MIN_TX, N_NULL=200):
    suffix = "" if MIN_TX == 2 else f"_min{MIN_TX}"   # main spec is MIN_TX=2; others are sensitivity runs
    rng = np.random.default_rng(5205)

    links = pd.read_csv(NETWORK / f"bipartite_edges_min{MIN_TX}.csv")
    product_nodes = pd.read_csv(NETWORK / f"product_nodes_min{MIN_TX}.csv").set_index("product")
    proximity_edges = pd.read_csv(NETWORK / f"product_edges_min{MIN_TX}.csv")

    # turn a list of communities into one label per product
    def labels(communities, order):
        node_to_community = {node: i for i, community in enumerate(communities) for node in community}
        return np.array([node_to_community[node] for node in order])

    products = sorted(links["product"].unique())
    network = product_graph(proximity_edges, products)
    (communities, modularity_q), runs = best_louvain(network, range(50))
    # order communities by size, name them C1..Ck
    communities = sorted(communities, key=len, reverse=True)
    community_labels = labels(communities, products)
    disease_areas = product_nodes.loc[products, "area"].values
    # seed stability: how much the first 10 Louvain runs agree
    seed_stability = np.mean([adjusted_rand_score(labels(runs[i], products), labels(runs[j], products))
                    for i in range(10) for j in range(i + 1, 10)])
    # how closely communities match disease areas
    area_nmi, area_ari = normalized_mutual_info_score(disease_areas, community_labels), adjusted_rand_score(disease_areas, community_labels)
    print(f"observed: {len(communities)} communities, modularity Q={modularity_q:.3f}, seed stability ARI={seed_stability:.2f}")
    print(f"alignment with disease area: NMI={area_nmi:.2f}, ARI={area_ari:.2f}")

    # ---- null model: degree-preserving bipartite rewiring
    null_draws = []
    for draw in range(N_NULL):
        null_links = rewire_links(links, 10 * len(links), rng)
        null_network = product_graph(project_products(null_links), products)
        (null_communities, null_q), _ = best_louvain(null_network, rng.integers(1e6, size=3))
        null_labels = labels(null_communities, products)
        null_draws.append((null_q, len(null_communities), normalized_mutual_info_score(disease_areas, null_labels)))
    null_draws = pd.DataFrame(null_draws, columns=["Q", "k", "nmi"])
    null_draws.to_csv(NETWORK / f"null_model_min{MIN_TX}.csv", index=False)
    z_modularity = (modularity_q - null_draws.Q.mean()) / null_draws.Q.std()
    z_nmi = (area_nmi - null_draws.nmi.mean()) / null_draws.nmi.std()
    p_modularity = (null_draws.Q >= modularity_q).mean()
    p_nmi = (null_draws.nmi >= area_nmi).mean()
    print(f"null ({N_NULL} rewirings): Q mean {null_draws.Q.mean():.3f} sd {null_draws.Q.std():.3f} -> z={z_modularity:.1f}, p={p_modularity:.3f}")
    print(f"null NMI mean {null_draws.nmi.mean():.2f} sd {null_draws.nmi.std():.2f} -> z={z_nmi:.1f}, p={p_nmi:.3f}")

    # ---- centrality
    distances = {(u, v): 1 - d["weight"] + 1e-6 for u, v, d in network.edges(data=True)}
    nx.set_edge_attributes(network, distances, "dist")
    centrality = pd.DataFrame({
        "community": [f"C{l+1}" for l in community_labels],
        "strength": pd.Series(dict(network.degree(weight="weight")))[products].values,
        "degree": pd.Series(dict(network.degree()))[products].values,
        "betweenness": pd.Series(nx.betweenness_centrality(network, weight="dist"))[products].values,
        "eigenvector": pd.Series(nx.eigenvector_centrality_numpy(
            network.subgraph(max(nx.connected_components(network), key=len)), weight="weight")).reindex(products).fillna(0).values,
    }, index=products)
    analyzed_nodes = product_nodes.join(centrality, how="inner")
    analyzed_nodes["core"] = analyzed_nodes.strength >= analyzed_nodes.strength.quantile(0.75)
    analyzed_nodes.round(4).to_csv(NETWORK / f"product_nodes_analyzed_min{MIN_TX}.csv")

    # ---- tables
    community_table = pd.crosstab(analyzed_nodes.community, analyzed_nodes.area)
    community_table["n_products"] = community_table.sum(axis=1)
    community_table["spend_usd_m"] = analyzed_nodes.groupby("community").spend_usd.sum().div(1e6).round(0)
    community_table["median_firms"] = analyzed_nodes.groupby("community").n_firms.median()
    community_table["top_products"] = analyzed_nodes.sort_values("spend_usd", ascending=False).groupby("community").apply(
        lambda d: "; ".join(d.index[:3]), include_groups=False)
    firm_communities = links.merge(analyzed_nodes[["community"]], left_on="product", right_index=True)
    community_table["top_firms"] = firm_communities.groupby(["community", "firm"]).spend_usd.sum().reset_index().sort_values(
        "spend_usd", ascending=False).groupby("community").apply(lambda d: "; ".join(d.firm[:3]), include_groups=False)
    community_table = community_table.sort_index(key=lambda s: s.str[1:].astype(int))
    community_table.to_csv(TABLES / f"t7_communities{suffix}.csv")
    pd.set_option("display.width", 250, "display.max_colwidth", 70)
    print("\n", community_table.to_string())

    network_summary = pd.DataFrame([
        ("products", len(products)), ("edges", network.number_of_edges()), ("density", round(nx.density(network), 3)),
        ("communities", len(communities)), ("modularity_Q", round(modularity_q, 3)), ("seed_stability_ARI", round(seed_stability, 2)),
        ("null_Q_mean", round(null_draws.Q.mean(), 3)), ("null_Q_sd", round(null_draws.Q.std(), 3)), ("z_Q", round(z_modularity, 1)), ("p_Q", p_modularity),
        ("NMI_area", round(area_nmi, 2)), ("ARI_area", round(area_ari, 2)), ("null_NMI_mean", round(null_draws.nmi.mean(), 2)),
        ("z_NMI", round(z_nmi, 1)), ("p_NMI", p_nmi)], columns=["metric", "value"])
    network_summary.to_csv(TABLES / f"t8_network_summary{suffix}.csv", index=False)

    most_central = analyzed_nodes.sort_values("strength", ascending=False)
    most_central[["area", "community", "n_firms", "strength", "betweenness", "spend_usd"]].head(15).round(3).to_csv(TABLES / f"t9_central_products{suffix}.csv")
    print("\nmost central products:\n", most_central[["area", "community", "n_firms", "strength", "betweenness"]].head(10).round(2).to_string())
    print("\ncorr(strength, n_firms) =", round(analyzed_nodes.strength.corr(analyzed_nodes.n_firms, method="spearman"), 2),
          "| corr(strength, spend) =", round(analyzed_nodes.strength.corr(analyzed_nodes.spend_usd, method="spearman"), 2))
    print("core products by area:", analyzed_nodes[analyzed_nodes.core].area.value_counts().to_dict())
    print("median n_firms core vs periphery:", analyzed_nodes.groupby("core").n_firms.median().to_dict())

## Main spec

In [ ]:
build_network(2)
analyze_network(2)

With 2+ transactions required for a link, the network had 97 firms, 122 products and 491 firm-product links. The product network had 2,876 weighted edges and a density of 0.39.

Louvain found 7 communities (Q = 0.31), and they were stable across seeds (mean ARI 0.87). Modularity in the 200 rewired networks averaged 0.22 (sd 0.018), so the observed value was well above chance (z = 5.2). The rewiring keeps each firm's and product's number of links, so this structure did not come from firm size alone.

Communities matched disease areas only partly. NMI was 0.49, compared with a null mean of 0.07 (z = 19.5).

Core products, meaning the top quartile by network strength, were all HIV (18) or TB (13), with no malaria. They had a median of 7 suppliers, compared with 2 in the periphery. Strength correlated with the number of suppliers (Spearman 0.65) and with spend (0.55).

## Sensitivity

In [ ]:
build_network(1)
analyze_network(1)

When I counted a single transaction as a link, the network kept 7 communities and the same modularity (Q = 0.31, z = 5.8). NMI with disease area fell to 0.36, which makes sense because one-off purchases add links across areas. I kept 2+ transactions as the main spec because the community structure held and these one-off links were dropped.

## Figures

In [ ]:
set_style(grid=False)


product_nodes = pd.read_csv(NETWORK / "product_nodes_analyzed_min2.csv", index_col=0)
proximity_edges = pd.read_csv(NETWORK / "product_edges_min2.csv")
network = product_graph(proximity_edges, product_nodes.index)

# backbone: maximum spanning tree (per component) + all ties with phi >= 0.5
# keep the strongest ties so the plot stays readable
backbone = nx.maximum_spanning_tree(network, weight="weight")
backbone.add_edges_from((node_u, node_v, edge) for node_u, node_v, edge in network.edges(data=True) if edge["weight"] >= 0.5)
# community layout: lay out each community on its own, then place the blocks side by side,
# so the shaded regions are faithful and node color (disease area) shows misfits inside each block
ANCHOR = {"C1": (0.0, 0.0, 1.0), "C2": (2.75, 0.0, 1.0), "C3": (0.9, -2.55, 0.8),
          "C4": (3.0, -2.2, 0.4), "C5": (3.95, -3.05, 0.1), "C6": (4.35, -3.05, 0.04), "C7": (4.55, -3.05, 0.04)}
# position every product inside its community's block
pos = {}
for community, (anchor_x, anchor_y, box_size) in ANCHOR.items():
    members = product_nodes.index[product_nodes.community == community]
    if len(members) > 1:
        distance_graph = nx.complete_graph(members)                      # distance = 1 - phi (1.2 if no tie)
        for node_u, node_v in distance_graph.edges:
            distance_graph[node_u][node_v]["d"] = 1.05 - network[node_u][node_v]["weight"] if network.has_edge(node_u, node_v) else 1.2
        layout = nx.kamada_kawai_layout(distance_graph, weight="d")
    else:
        layout = {members[0]: np.zeros(2)}
    coords = np.array(list(layout.values()))
    coords = coords - np.median(coords, 0)
    scale = np.quantile(np.abs(coords), 0.95) or 1
    coords = np.clip(coords / scale, -1, 1)
    for node, point in zip(layout, coords * box_size):
        pos[node] = point + np.array([anchor_x, anchor_y])

COMM_NAME = {"C1": "C1  HIV generics core", "C2": "C2  TB generics",
             "C3": "C3  Malaria, injectables & originators", "C4": "C4  TB patient kits",
             "C5": "C5-C7  single-supplier\nproducts"}
fig, ax = plt.subplots(figsize=(10, 8.6))
for community, (anchor_x, anchor_y, box_size) in ANCHOR.items():
    if community in ("C6", "C7"): continue
    half_width = box_size + 0.18 if community != "C5" else 0.5
    center_x = anchor_x if community != "C5" else 4.25
    ax.add_patch(matplotlib.patches.FancyBboxPatch((center_x - half_width, anchor_y - box_size - 0.18), 2 * half_width, 2 * box_size + 0.36,
                 boxstyle="round,pad=0.02,rounding_size=0.12", fc="#f4f3ef", ec="#d6d5cf", lw=0.8, zorder=0))
    ax.text(center_x - half_width + 0.05, anchor_y + box_size + 0.22, COMM_NAME[community], fontsize=9.5, fontweight="bold", color=INK2, va="bottom")
# draw ties between and within communities
cross_edges = [(node_u, node_v, edge) for node_u, node_v, edge in backbone.edges(data=True) if product_nodes.community[node_u] != product_nodes.community[node_v]]
inner_edges = [(node_u, node_v, edge) for node_u, node_v, edge in network.edges(data=True) if product_nodes.community[node_u] == product_nodes.community[node_v] and edge["weight"] >= 0.25]
for node_u, node_v, edge in cross_edges:
    ax.plot(*zip(pos[node_u], pos[node_v]), color="#c9c8c2", lw=0.6, zorder=1, alpha=0.6, ls=":")
for node_u, node_v, edge in inner_edges:
    ax.plot(*zip(pos[node_u], pos[node_v]), color="#b9b8b2", lw=0.3 + 1.4 * edge["weight"], zorder=1, alpha=0.55)
node_sizes = 18 + 700 * np.sqrt(product_nodes.spend_usd / product_nodes.spend_usd.max())
ax.scatter([pos[node][0] for node in product_nodes.index], [pos[node][1] for node in product_nodes.index], s=node_sizes,
           c=product_nodes.area.map(AREA_COLORS), edgecolors="white", linewidths=1, zorder=2)
# shorten long product names for labels
shorten = lambda s: s.replace(" - FDC", "").replace("Dolutegravir", "DTG").replace("Lamivudine", "3TC") \
    .replace("Tenofovir DF", "TDF").replace("Tenofovir", "TDF").replace("Efavirenz", "EFV") \
    .replace("Zidovudine", "AZT").replace("Nevirapine", "NVP").replace("Emtricitabine", "FTC") \
    .replace("Ethambutol+Isoniazid+Pyrazinamide+Rifampicin (RHZE)", "RHZE").replace("Isoniazid", "INH") \
    .replace("Rifampicin", "RIF").replace(" + ", "+")[:30]
labeled = (product_nodes.sort_values("spend_usd", ascending=False).groupby("community").head(2).index.tolist()
       + ["Artemether+Lumefantrine - FDC", "Moxifloxacin", "Atazanavir+Ritonavir - FDC", "Amikacin"])
for i, node in enumerate(dict.fromkeys(labeled)):
    if product_nodes.community[node] in ("C5", "C6", "C7"): continue
    node_x, node_y = pos[node]
    ax.annotate(shorten(node), (node_x, node_y), xytext=(7, 5 if i % 2 else -9), textcoords="offset points", fontsize=7.5,
                color=INK, bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85), zorder=3)
for area, color in AREA_COLORS.items():
    ax.scatter([], [], s=60, c=color, label=area, edgecolors="white")
ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0.0, 0.02), fontsize=9,
          title="Node color: disease area", title_fontsize=8.5)
ax.set_aspect("equal")
ax.set_axis_off()
add_title(ax, "The product space of Global Fund medicines",
      "Boxes = network communities (Louvain). Products are close when the same firms supply both; node size = spend")
n_firms = pd.read_csv(NETWORK / "bipartite_edges_min2.csv").firm.nunique()
fig.text(0.02, 0.02, "Solid lines: ties within a community with proximity >= 0.25; dotted: strongest ties between communities (maximum spanning tree).\n"
         f"{len(product_nodes)} products, {n_firms} firms; a firm-product link requires 2+ PQR transactions. Source: Global Fund PQR, 2005-2026.", fontsize=7.5, color=MUTED)
save_figure(fig, "f5_product_space.png")

The main communities were:

- C1 HIV generics core: 44 products, 86% HIV, USD 9.0 bn (Mylan, Hetero, Cipla)
- C2 TB generics: 43 products, 93% TB, USD 1.9 bn (Macleods, Lupin, Mylan)
- C3 Malaria, injectables and originators: 24 products, 67% malaria, USD 0.7 bn (Guilin, Sanofi, Ipca)
- C4 TB patient kits: 7 products (Svizera, Labesfal, B. Braun)
- C5-C7: fosamprenavir and maraviroc, arteether, and pyronaridine-artesunate, each with very few suppliers

I chose these labels from each community's main disease area and top suppliers, so they are my interpretation.

The misfits were the most informative part. Artemether-lumefantrine, the largest malaria product (USD 0.9 bn), sat in the HIV core. Its suppliers include Cipla, Mylan, Macleods and Strides, which all work across disease areas. Moxifloxacin, levofloxacin and delamanid also sat in the HIV core even though they are TB products. Amikacin and meropenem (TB) and older ARVs such as saquinavir and nelfinavir sat with malaria in C3. I read this as a sign that firm capabilities organize the network, not only disease area.

The network pools 2005-2026, and each PQR product name is a separate node, so pediatric and adult forms count as different products.

In [ ]:
# f6 null model
null_draws = pd.read_csv(NETWORK / "null_model_min2.csv")
summary = pd.read_csv(TABLES / "t8_network_summary.csv").set_index("metric")["value"].astype(float)
fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.4))
for ax, metric, observed, axis_label in [(axes[0], "Q", summary["modularity_Q"], "Modularity (Q)"),
                           (axes[1], "nmi", summary["NMI_area"], "Agreement with disease areas (NMI)")]:
    ax.hist(null_draws[metric], bins=25, color="#B4B5B8", edgecolor="white", linewidth=0.8)
    ax.axvline(observed, color=AREA_COLORS["HIV/ARV"], lw=2)
    ax.text(observed, ax.get_ylim()[1] * 0.92, f"  observed\n  {observed:.2f}", color=INK, fontsize=8.5, va="top")
    ax.set_xlabel(axis_label)
    ax.set_ylabel("Null draws")
    ax.grid(axis="y", color=GRID, lw=0.6)
    ax.set_xlim(min(null_draws[metric].min(), observed) * 0.9 if metric == "Q" else 0, observed * 1.25)
axes[0].set_title("Community structure is not an artefact of firm size", loc="left", fontsize=11, fontweight="bold", pad=10)
fig.text(0.01, -0.06, f"Gray: {len(null_draws)} re-wired networks that keep each firm's and product's number of links. "
         f"Observed values exceed every draw (p < {1 / len(null_draws):.3f}).", fontsize=7.5, color=MUTED)
fig.tight_layout()
save_figure(fig, "f6_null_model.png")

In [ ]:
# f7 composition of communities
community_table = pd.read_csv(TABLES / "t7_communities.csv", index_col=0)
community_table = community_table[community_table.n_products >= 3]
area_shares = community_table[list(AREA_COLORS)].div(community_table.n_products, axis=0)[::-1]
row_labels = [COMM_NAME.get(c, c).replace("\n", " ") + f"  (n={int(community_table.loc[c, 'n_products'])})" for c in area_shares.index]
fig, ax = plt.subplots(figsize=(8, 2.8))
bar_start = np.zeros(len(area_shares))
for area, color in AREA_COLORS.items():
    ax.barh(row_labels, area_shares[area] * 100, left=bar_start, color=color, height=0.6, edgecolor="white", linewidth=2, label=area)
    for i, value in enumerate(area_shares[area]):
        if value >= 0.08: ax.text(bar_start[i] + value * 50, i, f"{value:.0%}", ha="center", va="center", color=INK if area == "Malaria" else "white", fontsize=8.5, fontweight="bold")
    bar_start += area_shares[area].values * 100
ax.set_xlim(0, 100)
ax.set_xlabel("Share of products in community (%)")
ax.tick_params(axis="y", length=0)
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.4, 1.0), fontsize=9)
ax.set_title("Communities mostly follow disease areas, with telling exceptions", loc="left", fontsize=11, fontweight="bold", pad=28)
save_figure(fig, "f7_community_composition.png")